In [2]:
# %% [markdown]
# # Modelo SARIMAX — Previsão de Visualizações do Portal TRT18
# Pipeline uniforme ao do ARIMA: mesma carga de dados, split sequencial 816/204,
# teste ADF, seleção por AIC, previsão em bloco único de 204 passos e mesmas métricas.
# Diferença em relação ao ARIMA: acrescenta componente sazonal (m=7) e variáveis
# exógenas de calendário (o "S" e o "X").
 
# %% Imports e configuração
import os
import time
import itertools
import warnings
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
 
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,
)
 
CAMINHO_CSV      = "../dados/trafego_tratado.csv"   
PASTA_RESULTADOS = "../resultados"
ALVO             = "Visualizações"
N_TESTE          = 204   # mesmo holdout do ARIMA (816 treino / 204 teste)
M_SAZONAL        = 7    # ciclo semanal
 
# A estrutura semanal já é capturada pelo termo sazonal (m=7) e pelas
# dummies de dia da semana, então p/q não precisam ser altos como no ARIMA puro.
GRID_P     = [0, 1, 2]
GRID_Q     = [0, 1, 2]
GRID_P_SAZ = [0, 1]
GRID_Q_SAZ = [0, 1]
GRID_D_SAZ = [0, 1]
# --------------------------------------------------------------------------
 
os.makedirs(PASTA_RESULTADOS, exist_ok=True)
warnings.filterwarnings("ignore")   # silencia avisos de convergência durante a grade
 
 
# ──────────────────────────────────────────────
# Função de plotagem padronizada
# ──────────────────────────────────────────────
def plot_previsao(
    y_real,
    y_prev,
    titulo,
    caminho_saida,
    ic_inferior=None,
    ic_superior=None,
):
    """
    Gera gráfico padronizado: Real (azul) x Previsto (laranja).
    Parâmetros
    ----------
    y_real       : pd.Series com índice DatetimeIndex
    y_prev       : pd.Series com índice DatetimeIndex
    titulo       : str — ex. 'SARIMAX(1,0,2)(1,1,1,7) — Valores reais x previstos'
    caminho_saida: str — caminho completo para o PNG de saída
    ic_inferior  : pd.Series opcional — limite inferior do IC 95%
    ic_superior  : pd.Series opcional — limite superior do IC 95%
    """
    fig, ax = plt.subplots(figsize=(13, 5))
 
    ax.plot(y_real.index, y_real.values,
            color="#1f77b4", linewidth=1.2, label="Real")
    ax.plot(y_prev.index, y_prev.values,
            color="#ff7f0e", linewidth=1.2, label="Previsto")
 
    if ic_inferior is not None and ic_superior is not None:
        ax.fill_between(
            y_prev.index,
            ic_inferior.values,
            ic_superior.values,
            color="#ff7f0e",
            alpha=0.12,
            label="IC 95%",
        )
 
    # Eixo X: ticks mensais, formato YYYY-MM (igual ao gráfico do ARIMA)
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=0, ha="center")
 
    # Grade horizontal cinza claro (idêntica ao estilo do ARIMA)
    ax.yaxis.grid(True, color="#cccccc", linewidth=0.7, linestyle="-")
    ax.set_axisbelow(True)
    ax.xaxis.grid(False)
 
    ax.set_title(titulo, fontsize=11)
    ax.set_xlabel("Data")
    ax.set_ylabel("Visualizações")
    ax.legend(loc="upper right")
 
    plt.tight_layout()
    plt.savefig(caminho_saida, dpi=150)
    plt.close(fig)
    print(f"Gráfico salvo: {caminho_saida}")
 
 
# ──────────────────────────────────────────────
# Função de métricas
# ──────────────────────────────────────────────
def avaliar(y_real, y_prev):
    return {
        "MAE"    : mean_absolute_error(y_real, y_prev),
        "RMSE"   : float(np.sqrt(mean_squared_error(y_real, y_prev))),
        "MAPE_%" : mean_absolute_percentage_error(y_real, y_prev) * 100,
        "R2"     : r2_score(y_real, y_prev),
    }
 
 
# %% Carga e preparo da série
df = pd.read_csv(CAMINHO_CSV, encoding="utf-8")
df["Data"] = pd.to_datetime(df["Data"])
df = df.set_index("Data").sort_index().asfreq("D")
 
assert df.isna().sum().sum() == 0, "Há datas faltantes/NaN após asfreq('D')."
print(f"Observações: {len(df)} | de {df.index.min().date()} a {df.index.max().date()}")
 
# %% Exógenas
# Exclusões deliberadas:
#   'fim_de_semana'       → combinação linear das dummies (multicolinearidade perfeita)
#   'Sessões','Usuários'  → contemporâneas ao alvo, causam vazamento de dados
#   'mes'                 → efeito menor, já coberto por recesso_judiciario
EVENTOS = [
    "recesso_judiciario",
    "feriado_nacional_fixo",
    "carnaval",
    "quarta_cinzas",
    "sexta_paixao",
    "corpus_christi",
    "data_especifica_judiciario",
    "ponto_facultativo_emenda",
]
 
# drop_first=True evita dummy trap (colinearidade exata entre as 7 dummies)
dow = pd.get_dummies(df["dia_semana"], prefix="dow", drop_first=True).astype(float)
X   = pd.concat([dow, df[EVENTOS].astype(float)], axis=1)
y   = df[ALVO].astype(float)
 
print(f"Exógenas ({X.shape[1]}): {list(X.columns)}")
 
# %% Split sequencial treino / teste
y_treino, y_teste = y.iloc[:-N_TESTE], y.iloc[-N_TESTE:]
X_treino, X_teste = X.iloc[:-N_TESTE], X.iloc[-N_TESTE:]
print(f"Treino: {len(y_treino)} | Teste: {len(y_teste)}")
 
# %% Estacionariedade (ADF) — define d
adf_stat, adf_p, *_ = adfuller(y_treino.dropna())
d = 0 if adf_p < 0.05 else 1
print(f"ADF: estatística={adf_stat:.3f} | p-valor={adf_p:.4f} -> d={d}")
 
# %% Busca em grade por AIC
melhor      = {"aic": np.inf, "order": None, "sorder": None, "res": None}
combinacoes = list(itertools.product(GRID_P, GRID_Q, GRID_P_SAZ, GRID_D_SAZ, GRID_Q_SAZ))
print(f"Testando {len(combinacoes)} configurações...")
 
t0 = time.time()
for i, (p, q, P, D, Q) in enumerate(combinacoes, 1):
    try:
        modelo = SARIMAX(
            y_treino,
            exog=X_treino,
            order=(p, d, q),
            seasonal_order=(P, D, Q, M_SAZONAL),
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        res = modelo.fit(disp=False, maxiter=100)
        if np.isfinite(res.aic) and res.aic < melhor["aic"]:
            melhor.update(aic=res.aic, order=(p, d, q),
                          sorder=(P, D, Q, M_SAZONAL), res=res)
    except Exception:
        continue
    if i % 12 == 0:
        print(f"  {i}/{len(combinacoes)} | melhor AIC: {melhor['aic']:.1f}")
 
print(f"Tempo da busca: {time.time() - t0:.1f}s")
print(f"Configuração selecionada: SARIMAX{melhor['order']}{melhor['sorder']} | AIC={melhor['aic']:.1f}")
res = melhor["res"]
 
# %% Previsão — bloco único de 204 passos (uniforme com o ARIMA)
fc        = res.get_forecast(steps=N_TESTE, exog=X_teste)
pred      = fc.predicted_mean
ic        = fc.conf_int(alpha=0.05)
metricas  = avaliar(y_teste, pred)
 
print("\n--- Métricas no conjunto de teste (bloco único) ---")
print(f"MAE {metricas['MAE']:.0f} | RMSE {metricas['RMSE']:.0f} | "
      f"MAPE {metricas['MAPE_%']:.1f}% | R² {metricas['R2']:.3f}")
 
# %% Salvar CSV de previsões
df_prev = pd.DataFrame({"real": y_teste, "previsto": pred})
df_prev.to_csv(
    os.path.join(PASTA_RESULTADOS, "sarimax_previsoes.csv"),
    encoding="utf-8-sig",
)
 
# %% Salvar CSV de métricas
df_met = pd.DataFrame([metricas])
df_met.insert(0, "modelo",  f"SARIMAX{melhor['order']}{melhor['sorder']}")
df_met.insert(1, "protocolo", "bloco_unico_204")
df_met.to_csv(
    os.path.join(PASTA_RESULTADOS, "sarimax_metricas.csv"),
    encoding="utf-8-sig",
    index=False,
)
 
# %% Gráfico padronizado
titulo = (
    f"SARIMAX{melhor['order']}{melhor['sorder']}"
    f" — Valores reais x previstos"
)
plot_previsao(
    y_real        = y_teste,
    y_prev        = pred,
    titulo        = titulo,
    caminho_saida = os.path.join(PASTA_RESULTADOS, "sarimax_previsao.png"),
    # IC omitido intencionalmente: em bloco único de 204 passos o intervalo
    # se abre muito (±20-70k) e polui o gráfico sem acrescentar informação útil.
    # Descomente as linhas abaixo se quiser incluí-lo:
    # ic_inferior = ic.iloc[:, 0],
    # ic_superior = ic.iloc[:, 1],
)
 
print(f"\nArquivos salvos em: {os.path.abspath(PASTA_RESULTADOS)}")
print("  - sarimax_previsoes.csv")
print("  - sarimax_metricas.csv")
print("  - sarimax_previsao.png")

Observações: 1020 | de 2023-07-01 a 2026-04-15
Exógenas (14): ['dow_2', 'dow_3', 'dow_4', 'dow_5', 'dow_6', 'dow_7', 'recesso_judiciario', 'feriado_nacional_fixo', 'carnaval', 'quarta_cinzas', 'sexta_paixao', 'corpus_christi', 'data_especifica_judiciario', 'ponto_facultativo_emenda']
Treino: 816 | Teste: 204
ADF: estatística=-3.982 | p-valor=0.0015 -> d=0
Testando 72 configurações...
  12/72 | melhor AIC: 15843.3
  24/72 | melhor AIC: 15787.5
  36/72 | melhor AIC: 15787.5
  48/72 | melhor AIC: 15750.6
  60/72 | melhor AIC: 15750.6
  72/72 | melhor AIC: 15750.6
Tempo da busca: 50.8s
Configuração selecionada: SARIMAX(1, 0, 2)(1, 1, 1, 7) | AIC=15750.6

--- Métricas no conjunto de teste (bloco único) ---
MAE 6488 | RMSE 10279 | MAPE 68.3% | R² 0.626
Gráfico salvo: ../resultados\sarimax_previsao.png

Arquivos salvos em: c:\Users\DIOGO\Desktop\Vs Code Git Hub\mp-Diogo\projeto\resultados
  - sarimax_previsoes.csv
  - sarimax_metricas.csv
  - sarimax_previsao.png
